# Lab 07: Multi-Agent with Agents as Tools

**Goal:** compose specialized agents. An orchestrator calls specialist agents the way it calls any other tool.

Course module: `samples/10-agents-as-tools`

Three multi-agent patterns in Strands:
1. **Agents as tools**: an orchestrator delegates to specialists (this lab, and the simplest)
2. **Graph**: explicit workflow with guaranteed order (course `11-graphs`)
3. **Swarm**: agents hand off to each other dynamically (course `12-swarms`)

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 1. Method 1: pass an agent directly in `tools`
Give the agent a `name` and `description`. The orchestrator sees it as a tool that takes a prompt.

In [ ]:
from strands import Agent, tool

@tool
def product_catalog(query: str) -> str:
    """Look up products in our catalog.

    Args:
        query: product keyword
    """
    return {
        "headphones": "Wireless Headphones Pro, $79.99, 30h battery, ANC",
        "keyboard": "Mechanical Keyboard RGB, $149.99, hot-swap switches",
    }.get(next((k for k in ["headphones", "keyboard"] if k in query.lower()), ""), "not found")

researcher = Agent(
    name="product_researcher",
    description="Looks up facts about our products. Give it a product question.",
    model=get_model(),
    system_prompt="You are a product researcher. Use the catalog tool and return only facts.",
    tools=[product_catalog],
    callback_handler=None,
)

writer = Agent(
    model=get_model(),
    system_prompt="You are a marketing copywriter. Always get facts from product_researcher before writing.",
    tools=[researcher],
)
print("writer tools:", writer.tool_names, "\n")
result = writer("Write a two-sentence product blurb for our wireless headphones.")

## 2. Method 2: wrap the specialist in a `@tool` function
This gives you more control: a custom signature, a **fresh agent per call** (no context bleed between calls), per-specialist models, and error handling. The course uses a bigger orchestrator model with cheaper specialists.

In [ ]:
@tool
def research_assistant(query: str, depth: str = "normal") -> str:
    """Research a product question and return findings.

    Args:
        query: The research question
        depth: How thorough: "quick", "normal", or "deep"
    """
    try:
        specialist = Agent(
            model=get_model(),            # e.g. a smaller/cheaper model in production
            system_prompt=f"You are a product researcher. Research depth: {depth}. Use the catalog tool.",
            tools=[product_catalog],
            callback_handler=None,
        )
        return str(specialist(query))
    except Exception as e:
        return f"Research failed: {e}"

orchestrator = Agent(
    model=get_model(),
    system_prompt="You are a sales assistant. Use research_assistant for product facts, then answer the customer.",
    tools=[research_assistant],
    callback_handler=None,
)
result = orchestrator("I'm torn between your headphones and your keyboard as a gift under $100. Which fits?")
print(result)

## 3. Where to go next
You now have the core of Strands: the loop, providers, tools and MCP, streaming, hooks, sessions, and multi-agent composition. The rest of the course builds on this:

| Course module | Topic |
|---|---|
| `06-plugins-skills` | Plugins and on-demand `AgentSkills` (SKILL.md) |
| `07-steering` | Steering handlers to enforce workflows |
| `08-context-management` | `context_manager="auto"`, summarizing, sliding windows |
| `11-graphs` / `12-swarms` | Structured and dynamic multi-agent systems |
| `13-evals` | `strands-agents-evals`: LLM judges, trajectories, red teaming |
| `14-deploy` | Bedrock AgentCore Runtime and Lambda |

- Docs: https://strandsagents.com/docs/user-guide/sdk/quickstart/python/
- Course repo: https://github.com/aws-samples/sample-building-with-strands-course